In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys


common_modules_path = "/content/drive/MyDrive/ai-projects/2026-03-transfer-learning-resnet-50"
sys.path.append(common_modules_path)
print(f"Updated sys.path: {sys.path}")

Updated sys.path: ['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/drive/MyDrive/ai-projects/2026-03-transfer-learning-resnet-50']


# PyTorch ResNet-50 Implementation

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from base_loader import BaseDataLoader
from torch_dataset import TorchMedicalDataset
import config
import json
import time
import numpy as np  # ✅ ADDED
import os

# Set seed for reproducibility
torch.manual_seed(config.SEED)

# At the VERY TOP of your script, before any CUDA operations
import torch
torch.backends.cudnn.benchmark = True  # ← ADD THIS


result_dir = config.PROJECT_ROOT_DIR + '/torch-results'
os.makedirs(result_dir, exist_ok=True)

## Load Dataset

In [ ]:
# ============================================
# Transforms
# ============================================
train_transform = transforms.Compose([
    transforms.Resize(config.IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Normalize(mean=config.MEAN, std=config.STD)
])

val_transform = transforms.Compose([
    transforms.Resize(config.IMG_SIZE),
    transforms.Normalize(mean=config.MEAN, std=config.STD)
])

# ============================================
# DataLoaders
# ============================================
train_base = BaseDataLoader(config.COVIDQU_PATH, split="Train", is_train_shuffle=True, seed=config.SEED)
val_base = BaseDataLoader(config.COVIDQU_PATH, split="Val", is_train_shuffle=False, seed=config.SEED)
test_base = BaseDataLoader(config.COVIDQU_PATH, split="Test", is_train_shuffle=False, seed=config.SEED)

train_ds = TorchMedicalDataset(train_base, transform=train_transform)
val_ds = TorchMedicalDataset(val_base, transform=val_transform)
test_ds = TorchMedicalDataset(test_base, transform=val_transform)

# Use pin_memory for faster GPU transfer
train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## Build Model

In [ ]:
# ============================================
# Model
# ============================================
def build_model(num_classes):
    model = models.resnet50(weights='ResNet50_Weights.IMAGENET1K_V1')

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Replace classifier head
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )
    return model

num_classes = len(train_base.class_to_idx)
model = build_model(num_classes)
model = model.to(config.DEVICE)  # ✅ ADDED

# Count trainable parameters (✅ MOVED here — after model is built)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

Trainable parameters: 525,315


## Loss, Optimizer, Scheduler

In [ ]:
# ============================================
# Loss, Optimizer, Scheduler
# ============================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config.LEARNING_RATE, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

## Training Functions

In [ ]:
# ============================================
# Training Functions
# ============================================
from tqdm import tqdm  # Install: pip install tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Add progress bar
    pbar = tqdm(loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        pbar.set_postfix({'loss': f'{running_loss/(total/len(loader)):.4f}'})

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

## Training Loop

In [ ]:
# Verify GPU is being used
print(f"Device: {config.DEVICE}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    # Clear cache
    torch.cuda.empty_cache()

Device: cuda
GPU available: True
GPU name: Tesla T4


In [ ]:
# ============================================
# Training Loop
# ============================================
best_val_acc = 0.0
history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'time': []}
start_time = time.time()

for epoch in range(config.EPOCHS):
    epoch_start = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, config.DEVICE)
    val_loss, val_acc = validate(model, val_loader, criterion, config.DEVICE)
    scheduler.step(val_loss)

    epoch_time = time.time() - epoch_start

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), result_dir + '/torch_best_model.pth')

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['time'].append(epoch_time)

    print(f"Epoch {epoch+1}/{config.EPOCHS} - {epoch_time:.1f}s - loss: {train_loss:.4f} - acc: {train_acc:.2f}% - val_loss: {val_loss:.4f} - val_acc: {val_acc:.2f}%")

train_time = time.time() - start_time

Training:   0%|          | 0/30 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training: 100%|██████████| 30/30 [00:27<00:00,  1.10it/s, loss=0.1296]


Epoch 1/15 - 34.8s - loss: 0.5370 - acc: 78.94% - val_loss: 0.5147 - val_acc: 81.87%


Training: 100%|██████████| 30/30 [00:30<00:00,  1.00s/it, loss=0.1256]


Epoch 2/15 - 37.5s - loss: 0.5201 - acc: 79.72% - val_loss: 0.4928 - val_acc: 81.65%


Training: 100%|██████████| 30/30 [00:28<00:00,  1.06it/s, loss=0.1200]


...

Epoch 13/15 - 36.6s - loss: 0.3892 - acc: 83.99% - val_loss: 0.4220 - val_acc: 83.48%


Training: 100%|██████████| 30/30 [00:27<00:00,  1.07it/s, loss=0.0898]


Epoch 14/15 - 36.8s - loss: 0.3718 - acc: 85.38% - val_loss: 0.3981 - val_acc: 84.87%


Training: 100%|██████████| 30/30 [00:29<00:00,  1.03it/s, loss=0.0860]


Epoch 15/15 - 36.5s - loss: 0.3562 - acc: 85.54% - val_loss: 0.3836 - val_acc: 84.98%


## Evaluation, Collect Predictions, Save Results, Summary

In [ ]:
# ============================================
# Evaluation
# ============================================
test_loss, test_acc = validate(model, test_loader, criterion, config.DEVICE)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

# ============================================
# Collect Predictions
# ============================================
def collect_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

y_true, y_pred, y_proba = collect_predictions(model, test_loader, config.DEVICE)

# ============================================
# Save Results
# ============================================
np.savez(result_dir + '/torch_predictions.npz', y_true=y_true, y_pred=y_pred, y_pred_proba=y_proba)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = {
    'framework': 'PyTorch',
    'test_accuracy': float(test_acc),
    'test_loss': float(test_loss),
    'training_time_seconds': train_time,

    'best_val_acc': float(best_val_acc),
    'epochs_completed': len(history['epoch'])

    'test_precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
    'test_recall_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
    'test_f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    'test_precision_weighted': float(precision_score(y_true, y_pred, average='weighted', zero_division=0)),
    'test_recall_weighted': float(recall_score(y_true, y_pred, average='weighted', zero_division=0)),
    'test_f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
    'num_samples': len(y_true),
    'class_names': class_names
}

with open(result_dir + '/torch_results.json', 'w') as f:
    json.dump(results, f, indent=4)

with open(result_dir + '/torch_history.json', 'w') as f:
    json.dump(history, f, indent=4)

print(f"\n✅ Results saved to {result_dir}/torch_results.json")

# ============================================
# Summary
# ============================================
print("\n=== PyTorch Training Complete ===")
print(f"Best Val Accuracy: {best_val_acc:.2f}%")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Total Time: {train_time:.1f} seconds")
print(f"Trainable Params: {trainable_params:,}")


Test Loss: 0.3205
Test Accuracy: 88.08%

✅ Results saved to /content/drive/MyDrive/ai-projects/2026-03-transfer-learning-resnet-50/torch-results/torch_results.json

=== PyTorch Training Complete ===
Best Val Accuracy: 85.30%
Test Accuracy: 88.08%
Total Time: 566.2 seconds
Trainable Params: 525,315


## Goal

Save model weights at best validation accuracy, log per-epoch metrics to JSON, compute full test metrics (accuracy, F1, confusion matrix), and save all predictions — then create comparison plots and tables to evaluate both frameworks side by side.